# NumPy Interview Problems

Common NumPy problems from data science and ML engineering interviews.

**Topics:** moving average, unique rows, most frequent value, binning, rolling windows, strides, missing value imputation, image operations, FFT basics.

## Problem 1: Moving Average

Implement a moving average (simple sliding window average) of window size k.

In [ ]:
import numpy as np

def moving_average(arr, window):
    cumsum = np.cumsum(arr, dtype=float)
    cumsum[window:] = cumsum[window:] - cumsum[:-window]
    return cumsum[window - 1:] / window

arr = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
print("arr:", arr)
ma = moving_average(arr, 3)
print("Moving avg (window=3):", ma)
# Expected: [2, 3, 4, 5, 6, 7, 8, 9]

# Verify
expected = np.array([np.mean(arr[i:i+3]) for i in range(len(arr)-2)])
print(f"Match: {np.allclose(ma, expected)}")

# Time complexity: O(n), Space: O(n)

## Problem 2: Find Unique Rows

Given a 2D array, find unique rows (case: deduplicate data matrix).

In [ ]:
def unique_rows(arr):
    # View each row as a tuple of bytes
    row_view = np.ascontiguousarray(arr).view(
        np.dtype((np.void, arr.dtype.itemsize * arr.shape[1]))
    )
    _, idx = np.unique(row_view, return_index=True)
    return arr[np.sort(idx)]

data = np.array([[1, 2, 3],
                 [4, 5, 6],
                 [1, 2, 3],   # duplicate
                 [7, 8, 9],
                 [4, 5, 6]])  # duplicate

print("Original:\n", data)
print("\nUnique rows:\n", unique_rows(data))

# Alternative using axis parameter (NumPy 1.13+)
print("\nUsing np.unique(axis=0):\n", np.unique(data, axis=0))

# Time complexity: O(n log n), Space: O(n)

## Problem 3: Most Frequent Value

Find the most frequent value in an array.

In [ ]:
def most_frequent(arr):
    values, counts = np.unique(arr, return_counts=True)
    max_idx = np.argmax(counts)
    return values[max_idx], counts[max_idx]

arr = np.array([1, 3, 2, 3, 4, 3, 1, 2, 3, 5, 3, 2, 1])
val, count = most_frequent(arr)
print(f"arr: {arr}")
print(f"Most frequent value: {val} (appears {count} times)")

# For integer arrays with limited range: bincount is faster
int_arr = np.array([1, 3, 2, 3, 4, 3, 1, 2, 3, 5])
most_common = np.argmax(np.bincount(int_arr))
print(f"Using bincount: {most_common}")

# Time complexity: O(n), Space: O(n)

## Problem 4: Binning Data

Digitize continuous data into bins and compute bin statistics.

In [ ]:
data = np.array([1.2, 3.4, 5.6, 7.8, 2.3, 4.5, 6.7, 8.9])
bins = np.array([0, 3, 6, 10])  # edges

# Which bin does each value fall into?
indices = np.digitize(data, bins)
print("Data:", data)
print("Bins:", bins)
print("Bin indices:", indices)

# Mean of values in each bin
bin_means = [np.mean(data[indices == i]) for i in range(1, len(bins))]
print("Bin means:", bin_means)

# Alternative: np.histogram with weights
counts, _ = np.histogram(data, bins)
sums, _ = np.histogram(data, bins, weights=data)
means = sums / counts
print("Bin means (via histogram):", np.where(np.isfinite(means), means, 0))

# Time complexity: O(n log n), Space: O(n)

## Problem 5: Rolling Window Operations

Generate all overlapping windows of size k.

In [ ]:
def rolling_windows(arr, window):
    return np.lib.stride_tricks.sliding_window_view(arr, window)

arr = np.arange(10)
windows = rolling_windows(arr, 3)
print("arr:", arr)
print("Rolling windows (k=3):\n", windows)

# Compute rolling sum
rolling_sum = np.sum(windows, axis=1)
print("Rolling sum:", rolling_sum)

# Rolling max
rolling_max = np.max(windows, axis=1)
print("Rolling max:", rolling_max)

# Manual implementation using strides (for understanding)
def rolling_strides(arr, window):
    shape = (arr.size - window + 1, window)
    strides = (arr.strides[0], arr.strides[0])
    return np.lib.stride_tricks.as_strided(arr, shape=shape, strides=strides)

print("\nUsing as_strided:\n", rolling_strides(arr, 3))

# Time complexity: O(n), Space: O(1) for view, O(n*k) if copied

## Problem 6: Missing Value Imputation

Replace NaN values with column mean.

In [ ]:
def impute_nan_with_mean(arr):
    arr = arr.copy().astype(float)
    col_mean = np.nanmean(arr, axis=0)
    nan_mask = np.isnan(arr)
    arr[nan_mask] = np.take(col_mean, np.where(nan_mask)[1])
    return arr

data = np.array([[1, 2, np.nan],
                 [4, np.nan, 6],
                 [7, 8, 9],
                 [np.nan, 11, 12]])

print("Original:\n", data)
imputed = impute_nan_with_mean(data)
print("\nImputed:\n", np.round(imputed, 2))

# Verify
expected_mean = np.nanmean(data, axis=0)
print("\nColumn means used:", np.round(expected_mean, 2))

# Time complexity: O(n), Space: O(n)

## Problem 7: Image as Array Operations

Simulate image processing: grayscale, flip, crop, threshold.

In [ ]:
# Simulate an RGB image (4x4 pixels, 3 channels)
img = np.random.randint(0, 256, (4, 4, 3), dtype=np.uint8)
print("Original image shape:", img.shape)
print("Image (first row):", img[0])

# Convert to grayscale (luminosity method)
gray = (0.2989 * img[:, :, 0] + 0.5870 * img[:, :, 1] +
        0.1140 * img[:, :, 2]).astype(np.uint8)
print("\nGrayscale:\n", gray)

# Flip horizontally
flipped = img[:, ::-1, :]
print("\nFlipped (first row):", flipped[0])

# Crop center 2x2
h, w = img.shape[:2]
crop = img[h//2-1:h//2+1, w//2-1:w//2+1, :]
print("\nCenter crop shape:", crop.shape)

# Threshold: values > 128 become 255
gray_threshold = np.where(gray > 128, 255, 0)
print("\nThresholded grayscale:\n", gray_threshold)

# Time complexity: O(n), Space: O(n)

## Problem 8: FFT Basics

Compute the frequency spectrum of a signal and find dominant frequencies.

In [ ]:
# Generate a signal: sum of two sine waves
fs = 1000  # sampling frequency
t = np.linspace(0, 1, fs, endpoint=False)
f1, f2 = 50, 120  # frequencies
signal = np.sin(2 * np.pi * f1 * t) + 0.5 * np.sin(2 * np.pi * f2 * t)

# Compute FFT
fft = np.fft.fft(signal)
freqs = np.fft.fftfreq(len(signal), 1/fs)

# Only positive frequencies
positive = freqs[:len(freqs)//2]
magnitude = np.abs(fft[:len(fft)//2])

# Find dominant frequencies
top_indices = np.argsort(magnitude)[-5:][::-1]
print("Dominant frequencies:")
for idx in top_indices:
    if magnitude[idx] > 1:
        print(f"  {positive[idx]:.1f} Hz (magnitude={magnitude[idx]:.1f})")

# Inverse FFT reconstruction
reconstructed = np.fft.ifft(fft).real
print(f"\nReconstruction error: {np.max(np.abs(signal - reconstructed)):.2e}")

# Time complexity: O(n log n), Space: O(n)

## Problem 9: K-Nearest Neighbors (Brute Force)

Find k-nearest neighbors for each point in a dataset.

In [ ]:
def knn(X, k=3):
    n = X.shape[0]
    # Compute all pairwise distances efficiently
    norms = np.sum(X**2, axis=1)
    dists = norms[:, np.newaxis] + norms[np.newaxis, :] - 2 * X @ X.T
    np.fill_diagonal(dists, np.inf)  # exclude self
    # Get k nearest
    nearest = np.argpartition(dists, k, axis=1)[:, :k]
    return nearest

X = np.array([[0, 0], [1, 1], [2, 2], [10, 10], [11, 11], [12, 12]])
neighbors = knn(X, k=2)

print("Data points:")
for i, pt in enumerate(X):
    print(f"  {i}: {pt}")
print("\n2-nearest neighbors (indices):")
for i in range(len(X)):
    print(f"  {i} -> {neighbors[i]}")

# Time complexity: O(n^2 * d + n * k), Space: O(n^2)

## Problem 10: Image Convolution (2D)

Apply a 3x3 kernel to a 2D image using sliding windows.

In [ ]:
def convolve2d(image, kernel):
    kh, kw = kernel.shape
    h, w = image.shape
    # Pad with zeros
    padded = np.pad(image, ((kh//2, kh//2), (kw//2, kw//2)), mode='constant')
    # Extract windows
    windows = np.lib.stride_tricks.sliding_window_view(padded, (kh, kw))
    # Apply kernel
    return np.sum(windows * kernel, axis=(-2, -1))

# Edge detection kernel (Sobel x)
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]])

image = np.array([
    [1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1],
    [1, 1, 0, 1, 1],   # vertical edge at center
    [1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1]
], dtype=float)

edges = convolve2d(image, sobel_x)
print("Original image:\n", image)
print("\nVertical edges:\n", np.round(edges, 1))

# Time complexity: O(n*m*kh*kw), Space: O(n*m)

## Problem 11: Find Local Maxima in 1D

Find all elements that are strictly greater than their neighbors.

In [ ]:
def local_maxima_1d(arr):
    # Compare with left and right neighbors
    left = arr[1:-1] > arr[:-2]
    right = arr[1:-1] > arr[2:]
    local_max = left & right
    indices = np.where(local_max)[0] + 1  # offset for slicing
    return indices, arr[indices]

arr = np.array([1, 3, 2, 5, 1, 4, 6, 2, 7, 3])
indices, values = local_maxima_1d(arr)

print("arr:", arr)
print(f"Local maxima at indices {indices}: values {values}")

# Verify
for idx, val in zip(indices, values):
    assert arr[idx-1] < val and arr[idx+1] < val, f"{idx} is not a peak"
print("All verified!")

# Time complexity: O(n), Space: O(n)

## Problem 12: Cumulative Sum with Reset

Compute cumulative sum that resets when a condition (e.g., zero) is encountered.

In [ ]:
def cumsum_reset(arr, reset_cond):
    # When reset_cond is True, start new sum from 0
    reset_idx = np.where(reset_cond)[0]
    if len(reset_idx) == 0:
        return np.cumsum(arr)
    # Use np.add.reduceat for segmented cumulative sums
    segments = np.split(arr, reset_idx[reset_idx > 0])
    return np.concatenate([np.cumsum(seg) for seg in segments])

arr = np.array([1, 2, 3, 0, 4, 5, 0, 6, 7, 8])
reset = arr == 0

result = cumsum_reset(arr, reset)
print("arr:", arr)
print("cumsum reset:", result)
# Expected: [1, 3, 6, 0, 4, 9, 0, 6, 13, 21]

# Time complexity: O(n), Space: O(n)

## Summary

Key patterns for interview problems:

| Problem | Approach | Complexity |
|---------|----------|------------|
| Moving average | `cumsum` diff or `sliding_window_view` | O(n) |
| Unique rows | View as void dtype or `axis=0` | O(n log n) |
| Most frequent | `unique(return_counts=True)` or `bincount` | O(n) |
| Binning | `digitize` or `histogram` | O(n log n) |
| Rolling windows | `sliding_window_view` or `as_strided` | O(n) |
| Missing values | `isnan` + `nanmean` + broadcasting | O(n) |
| FFT | `np.fft.fft` | O(n log n) |
| KNN | Broadcasting + `argpartition` | O(n²d) |
| Convolution | `sliding_window_view` + kernel | O(n·k²) |
| Local maxima | Compare with neighbors | O(n) |